# H06 — The matched filter on real data and simulated glitches

**Part VI · the laboratory of lecture 13, in class**

**Data.** `labs/data/gw150914_H1.hdf5`: the strain of the LIGO Hanford detector (H1), 32 s around the event GW150914 (GPS 1126259447 to 1126259479, merger at 1126259462.42), sampled at 4096 Hz in the file. The file for the Livingston detector (L1), used in A5, is `labs/data/gw150914_L1.hdf5`. Both come from the Gravitational Wave Open Science Center (GWOSC), release R1. `load_strain` reads them. The cell under the libraries names the event and downloads its files from GWOSC if they are not in `labs/data`, so the same laboratory can run on another event. The code high-passes the strain at 15 Hz and resamples it to 2048 Hz. The signals and the glitches are **simulated and injected**: they are models of what the sources and the instrument do and not recordings of them. With the files of GW150914 in `labs/data`, everything runs offline.

**Libraries.**

- `pycbc`: the time series, the matched filter, the PSD, the waveform models for the templates (`IMRPhenomD` and others) and the overlap `match`.
- `lal`: the waveform library that `pycbc` calls. The first cell switches off the redirection of its output.
- `numpy`: arrays and the FFT.
- `matplotlib.pyplot` and `matplotlib.animation`: the plots and the animations. `IPython.display` shows the animations in the page.
- `gwosc`, `gwpy` and `h5py`: the merger time of an event, the download of its public strain and the file where we save it.
- `gwcourse_signal` and `gwcourse_palette`: the functions of the course that load the strain and band-pass it, and the colors.
- `tomllib`: reads `labs/config.toml`, the configuration of the laboratories: the folder of the data, the events and their files.

> **References**
>
> (Usman et al. 2016), (LIGO Scientific Collaboration and Virgo Collaboration 2016)

## Tasks

**Part A. The demonstration.** We run the matched filter in the Hanford data and show what lecture 13 explained. All the code is written out and runs cell by cell. Under each cell we say what the result shows.

- **A1.** The noise spectrum and the template: the functions `condition`, `make_template` and `analyze` and the SNR expected for the 36 + 29 template at the distance of GW150914.
- **A2.** The filter on a signal injected in the Hanford data, with the template slid along the data: one template, then three templates in three panels.
- **A3.** The output against the Rayleigh distribution, on noise alone and with a signal.
- **A4.** Two simulated glitches and the output $\Phi$ of the filter for each.
- **A5.** Coincidence: the same template in Hanford and Livingston, for the event and for a glitch.
- **A6.** A claim to test: A4 repeated with a template of lower mass.

**Part B. Your turn.** Five functions are given: `load_data`, `make_template`, `inject_signal`, `matched_filter` and `plot_results`. We inject a signal and search for it.

- **B1.** Inject a signal with the masses and the distance you choose and find the SNR you expect.
- **B2.** Search for it with a template of the same masses and with a template of different masses.
- **B3.** Search for it with another waveform model and the same masses.
- **B4.** Decide whether the data allow a claim of detection.

In [ ]:
# libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# LAL redirects its printed output inside Jupyter and every waveform call becomes slow: switch it off
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False)

# the course root holds gwcourse_palette.py: find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)
sys.path.insert(0, os.path.join(here, 'labs'))

# the configuration of the laboratories: where the data are and the events we use
import tomllib
with open(os.path.join(here, 'labs', 'config.toml'), 'rb') as config_file:
    config = tomllib.load(config_file)
DATA_DIR = os.path.join(here, 'labs', config['data']['dir'])

# the course colors and the course function that reads the strain
from gwcourse_palette import P, apply_style
from gwcourse_signal import load_strain, bandpass
apply_style()

# the pycbc tools: time series, filters, PSD and waveforms.
# The filter of pycbc is imported under its own name: matched_filter is a function of Part B
from pycbc.types import TimeSeries
from pycbc.filter import matched_filter as pycbc_matched_filter
from pycbc.filter import highpass, resample_to_delta_t, sigma
from pycbc.psd import interpolate
from pycbc.waveform import get_td_waveform


## The data: the event and its files

The event of the laboratory is named in one place, `EVENT`. `download_strain` looks for the file of each detector in `labs/data`. For the events of `labs/config.toml` the file name and the merger time are read from there. If the file is not there, it downloads 32 s of public strain around the merger from GWOSC, at 4096 Hz, and saves it under the name the course uses, `<event>_<detector>.hdf5`. For GW150914 the two files are already there and nothing is downloaded.

To run the laboratory on another event, change `EVENT` to its name in the GWOSC catalogs, for example `'GW170814'`, and change the masses of the template in A1. The first run needs the network. For some events GWOSC serves only files of 4096 s, so the download takes about five minutes for each detector.


In [ ]:
# the tools that find and download the public data
import h5py
from gwosc.datasets import event_gps
from gwpy.timeseries import TimeSeries as GwpyTimeSeries

# the event of the laboratory and its two detectors
EVENT = 'GW150914'
DETECTORS = ('H1', 'L1')


def event_time(event=EVENT):
    """Give the GPS time of the merger of an event.

    Args:
        event: the name of the event in the GWOSC catalogs (default EVENT).

    Returns:
        gps: the GPS time of the merger (s).
    """
    # the time in the configuration, or the time in the GWOSC catalog (needs the network)
    if event in config['events']:
        gps = config['events'][event]['gps']
    else:
        gps = event_gps(event)
    return gps


def strain_path(event=EVENT, detector='H1'):
    """Give the file of the strain of one detector for one event.

    Args:
        event: the name of the event in the GWOSC catalogs (default EVENT).
        detector: 'H1', 'L1' or 'V1' (default 'H1').

    Returns:
        path: the file in the data folder.
    """
    # the file named in the configuration, or else the name the course uses, for example gw150914_H1.hdf5
    if detector in config['events'].get(event, {}):
        name = config['events'][event][detector]
    else:
        name = f"{event.lower()}_{detector}.hdf5"
    path = os.path.join(DATA_DIR, name)
    return path


def download_strain(event=EVENT, detector='H1', duration=32, fs=4096):
    """Download the strain around an event from GWOSC, if its file is not in labs/data.

    Args:
        event: the name of the event in the GWOSC catalogs (default EVENT).
        detector: 'H1', 'L1' or 'V1' (default 'H1').
        duration: the length of the record, with the merger in the middle (s, default 32).
        fs: the sampling frequency of the public data (Hz, default 4096).

    Returns:
        path: the file with the strain.
    """
    # the file of this event and detector
    path = strain_path(event, detector)
    name = os.path.basename(path)
    if os.path.exists(path):
        print(f"{name}: already in labs/data, nothing to download")
        return path
    # the stretch of data, with the merger in the middle
    start = int(event_time(event)) - duration // 2
    fetched = GwpyTimeSeries.fetch_open_data(detector, start, start + duration, sample_rate=fs)
    # save it in the layout of the GWOSC files, which load_strain reads
    with h5py.File(path, 'w') as f:
        strain = f.create_dataset('strain/Strain', data=fetched.value)
        strain.attrs['Xspacing'] = fetched.dt.value
        strain.attrs['Xstart'] = start
        f.create_dataset('meta/GPSstart', data=start)
        f.create_dataset('meta/Duration', data=duration)
    print(f"{name}: {duration} s downloaded from GWOSC")
    return path


def load_raw(detector='H1', f_high_pass=15.0, fs_out=2048.0, event=EVENT):
    """Read the strain of one detector around the event, high-pass it and resample it.

    Args:
        detector: 'H1' for Hanford or 'L1' for Livingston (default 'H1').
        f_high_pass: the frequency below which the strain is removed (Hz, default 15).
        fs_out: the sampling frequency after the resampling (Hz, default 2048).
        event: the name of the event (default EVENT).

    Returns:
        raw: the strain, a pycbc TimeSeries.
    """
    # the strain of the file, as a pycbc time series
    path = strain_path(event, detector)
    strain, times, fs = load_strain(path)
    series = TimeSeries(strain, delta_t=1.0 / fs, epoch=times[0])
    # remove the slow drift, then resample
    filtered = highpass(series, f_high_pass)
    raw = resample_to_delta_t(filtered, 1.0 / fs_out)
    return raw


# the merger time of the event and the files of the two detectors
GPS_EVENT = event_time(EVENT)
for detector in DETECTORS:
    download_strain(EVENT, detector)

# the strain of H1, high-passed at 15 Hz and resampled to 2048 Hz
raw = load_raw('H1')
print(f"{raw.duration} s of H1 at {1 / raw.delta_t:.0f} Hz, "
      f"{EVENT} at GPS {GPS_EVENT:.2f}")


## A1. The noise spectrum and the template

The matched filter of lecture 13 needs two things: the one-sided PSD $S_{n}(f)$ of the noise and a template. We define three functions and use them for the rest of the laboratory.

`condition(series)` removes two seconds from each end and estimates $S_{n}(f)$ with the Welch method on four second segments.

`make_template(mass1, mass2, distance=...)` builds the waveform of a binary with these masses, for a source at the distance in Mpc (1 Mpc if not given). The waveform model is `IMRPhenomD`: the inspiral, the merger and the ringdown of two black holes without spins. We use the polarization $h_{+}$ of a source seen face on, from 20 Hz. The template is as long as the data and has its merger at time zero. The argument `approximant` chooses another waveform model.

`analyze(series, template, psd)` returns the output $\Phi(t_{c})$ of the standard matched filter. The output is cropped, so that the transients at the edges of the record are outside what we look at.

With `make_template` we build the template of a 36 + 29 solar mass binary at 410 Mpc, the published distance of GW150914. For this template we compute $\sqrt{\langle h,h\rangle}$: the SNR that this binary, overhead and face on, would give in these data. With noise alone, $\sqrt{\langle h,h\rangle}$ is also the standard deviation of the filter output $\langle n,h\rangle$, so `pycbc` calls it `sigma`. The amplitude of the template does not change $\Phi$: the same template serves for the search.

In [ ]:
def condition(series, edge=2, seglen=4):
    """Crop the edges of the data and estimate the noise spectrum.

    Args:
        series: the data, a pycbc TimeSeries.
        edge: the time cropped at each end (s, default 2).
        seglen: the length of the Welch segments (s, default 4).

    Returns:
        d: the cropped data.
        psd: the one-sided PSD S_n(f) on the frequency grid of d.
    """
    # crop the ends, where the filters leave transients
    d = series.crop(edge, edge)
    # Welch estimate on segments of seglen seconds, then on the frequency grid of the data
    psd = interpolate(d.psd(seglen), d.delta_f)
    return d, psd


def make_template(mass1, mass2, f_low=20.0, approximant='IMRPhenomD', like=None, distance=1.0):
    """Build the template of a binary at a distance, as long as the data.

    Args:
        mass1, mass2: the masses (solar masses).
        f_low: the lower edge of the band (Hz, default 20).
        approximant: the waveform model (default 'IMRPhenomD'). Others that work
            here: 'IMRPhenomXAS', 'SEOBNRv4_opt', 'TaylorT4' (inspiral only).
        like: the time series whose sampling and length the template takes
            (default None: the conditioned data, `data`).
        distance: the distance of the source, overhead and face on (Mpc, default 1).

    Returns:
        shifted: the template, with the merger at time zero.
    """
    # the sampling and the length of the template are those of the data
    if like is None:
        like = data
    # the waveform at this distance, made as long as the data
    hp, _ = get_td_waveform(approximant=approximant, mass1=mass1, mass2=mass2,
                            delta_t=like.delta_t, f_lower=f_low, distance=distance)
    hp.resize(len(like))
    # the merger at time zero
    shifted = hp.cyclic_time_shift(hp.start_time)
    return shifted


def analyze(series, template, psd, f_low=20.0, crop=(6, 4)):
    """Run the matched filter on the data.

    Args:
        series: the data, a pycbc TimeSeries.
        template: the template, as long as the data.
        psd: the one-sided PSD S_n(f) of the noise.
        f_low: the lower edge of the band (Hz, default 20).
        crop: the time cropped at the start and at the end of the output (s, default 6 and 4).

    Returns:
        phi: the complex output Phi(t_c) of the filter.
    """
    # the filter output, without the transients at the edges
    phi = pycbc_matched_filter(template, series, psd=psd, low_frequency_cutoff=f_low)
    phi = phi.crop(crop[0], crop[1])
    return phi


# the conditioned data and the noise spectrum
data, psd = condition(raw)

# the template: 36 + 29 solar masses at the published distance of GW150914, waveform model IMRPhenomD
R_EVENT = 410.0                      # luminosity distance of GW150914 [Mpc]
template = make_template(36, 29, approximant='IMRPhenomD', distance=R_EVENT)
print(f"template: IMRPhenomD, 36 + 29 solar masses, no spins, h_plus face on, from 20 Hz, at {R_EVENT:.0f} Mpc")

# the SNR expected at this distance, for a source overhead and face on: sqrt(<h,h>)
rho_expected = sigma(template, psd=psd, low_frequency_cutoff=20)
print(f"expected SNR at {R_EVENT:.0f} Mpc, overhead and face on: {rho_expected:.1f}")


**What the result shows.** At 410 Mpc the expected SNR is 36.5. It is the SNR of GW150914 in Hanford if its source were overhead and face on. The event has $\Phi = 16.5$ in Hanford (A5), less than 36.5, because its source is not overhead and not face on.


## A2. The matched filter on a signal injected in the data

The filter of lecture 13 places the template at an arrival time $t_{c}$ and measures how well the template fits the data there. The result is $\Phi(t_{c})$. Sliding the template along the data gives the whole output.

We use the `data`, the `psd` and the `template` of A1. We add a signal to the data. The signal is the 36 + 29 template, at a known time and with $\varrho = 20$. The filter does not use the time.

### A signal in the data

We add the waveform to the data 8 seconds after the start of the record, far from GW150914. The function `inject` adds the waveform of a binary at 1 Mpc, multiplied by a factor. The factor is $\varrho_{\rm target}/\varrho_{1}$, with $\varrho_{1}$ the SNR of the same waveform at 1 Mpc, so that the SNR of the signal is 20. Then we look at the strain and at the band-passed data. The band-pass keeps the frequencies from 35 to 350 Hz, as in lecture 12 and in laboratory H05, and removes the rest, where the noise is largest. We apply the same band-pass to the data, to the signal and to the templates, so that they can be compared in the figures. The signal is drawn in brick on top of the data, so that its position is visible.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from pycbc.filter import match

# A signal in the data. We know the time of the merger. The filter does not.
T_DEMO = float(data.start_time) + 8.0     # time of the merger [s], away from the event
RHO_TARGET = 20.0                         # SNR of the injection
F_LOW = 20.0                              # lower edge of the band [Hz]


def inject(series, mass1, mass2, t_merger, scale, f_low=F_LOW, approximant='IMRPhenomD'):
    """Add the waveform of a binary to the data, with its merger at t_merger.

    Args:
        series: the data, a pycbc TimeSeries.
        mass1, mass2: the masses of the binary (solar masses).
        t_merger: the GPS time of the merger (s).
        scale: the factor that multiplies the waveform at 1 Mpc: the SNR wanted
            divided by the SNR at 1 Mpc, or 1 / distance for a distance in Mpc.
        f_low: the lower edge of the band (Hz, default 20).
        approximant: the waveform model (default 'IMRPhenomD').

    Returns:
        injected: the data with the signal added.
    """
    # the waveform at 1 Mpc, scaled
    wave, _ = get_td_waveform(approximant=approximant, mass1=mass1, mass2=mass2,
                              delta_t=series.delta_t, f_lower=f_low)
    wave = wave * scale
    # the merger at t_merger
    wave.start_time = float(wave.start_time) + t_merger
    injected = series.add_into(wave)
    return injected


def chirp_in_strain(mass1, mass2, t_merger, rho=RHO_TARGET, f_low=F_LOW):
    """Build the waveform of a binary with a chosen SNR, as long as the data.

    Args:
        mass1, mass2: the masses (solar masses).
        t_merger: the GPS time of the merger (s).
        rho: the SNR of the signal in the data (default 20).
        f_low: the lower edge of the band (Hz, default 20).

    Returns:
        placed: the waveform in strain, with its merger at t_merger.
    """
    # the SNR of the template at 1 Mpc gives the factor of the waveform
    rho_own = sigma(make_template(mass1, mass2, f_low), psd=psd, low_frequency_cutoff=f_low)
    scale = rho / rho_own
    # the waveform alone: injected in a series of zeros as long as the data
    empty = TimeSeries(np.zeros(len(data)), delta_t=data.delta_t, epoch=data.start_time)
    placed = inject(empty, mass1, mass2, t_merger, scale, f_low)
    return placed


def bandpass_series(series, f_low=35.0, f_high=350.0):
    """Band-pass a time series: keep the frequencies from f_low to f_high, with no phase shift.

    Args:
        series: the data, a pycbc TimeSeries.
        f_low, f_high: the edges of the band that is kept (Hz, default 35 and 350).

    Returns:
        filtered: the band-passed data, an array as long as series.
    """
    # the sampling frequency of the series
    fs = 1.0 / series.delta_t
    # the course filter: forwards and backwards, so that the signal does not move in time
    filtered = bandpass(np.asarray(series), fs, band=(f_low, f_high))
    return filtered

In [ ]:
# the signal, the data with the signal in it, and the band-passed versions of all three
signal = chirp_in_strain(36, 29, T_DEMO, rho=RHO_TARGET)
injected = data + signal
b_signal = bandpass_series(signal)
b_injected = bandpass_series(injected)
b_data = bandpass_series(data)
noise_rms = np.std(b_data)                     # the unit of the band-passed plots
t_rel = data.sample_times.numpy() - T_DEMO     # time from the injection [s]

# the strain, then the band-passed data: the signal in brick, on top of the data in gray
look = np.abs(t_rel) < 1.5
plt.figure(figsize=(9, 5))

# the strain
plt.subplot(2, 1, 1)
plt.plot(t_rel[look], np.asarray(injected)[look], color=P['GREY_MID'], lw=0.5,
         label='data with the signal')
plt.plot(t_rel[look], np.asarray(signal)[look], color=P['BRICK'], lw=1.2,
         label='the signal alone')
plt.ylabel('strain')
plt.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)

# the band-passed data, in rms of the band-passed noise
plt.subplot(2, 1, 2)
plt.plot(t_rel[look], b_injected[look] / noise_rms, color=P['GREY_MID'], lw=0.5)
plt.plot(t_rel[look], b_signal[look] / noise_rms, color=P['BRICK'], lw=1.2)
plt.ylabel('band-passed strain\n[rms of the noise]')
plt.xlabel('time from the injection [s]')
plt.show()

# the numbers of the plot
print(f"merger of the injection at GPS {T_DEMO:.2f} s, SNR "
      f"{sigma(signal, psd=psd, low_frequency_cutoff=F_LOW):.1f}")
print(f"strain: the signal peaks at {np.abs(np.asarray(signal)).max():.1e}, "
      f"the data have rms {np.std(np.asarray(data)):.1e}")
print(f"band-passed, in rms of the noise: the signal peaks at "
      f"{np.abs(b_signal[look]).max() / noise_rms:.1f}, "
      f"data with the signal reach {np.abs(b_injected[look]).max() / noise_rms:.1f}, "
      f"data alone reach {np.abs(b_data[look]).max() / noise_rms:.1f}")

**What the result shows.** In the strain the signal cannot be seen: its peak is $1.0\times10^{-21}$ against an rms of $1.3\times10^{-20}$ in the data. After the band-pass the data are dominated by the noise at the low end of the band. The signal peaks at 1.1 times the rms of the band-passed noise, and the data with the signal reach 3.1 against 2.9 for the same stretch of data alone. The chirp is visible only in the brick curve, which we know. In the data it cannot be told from the noise by eye: finding it is the work of the filter.

### The template, slid along the data

The filter uses the shape of the template and moves it in time. At each arrival time $t_{c}$ it multiplies the data by the template placed at $t_{c}$ and adds the products, giving to each frequency a weight that depends on the noise at that frequency (lecture 13). This sum is the correlation $z(t_{c}) = \langle s,h_{t_{c}}\rangle$ of lecture 13, divided by $\sqrt{\langle h,h\rangle}$. If the template lies on a signal, the products are mostly positive and the sum is large. Anywhere else the products have both signs and the sum stays small.

The cell computes the whole $\Phi(t_{c})$ with the filter of `pycbc`, the one that `analyze` uses. Then it prints $\Phi$ for two positions of the template: one second before the signal and on the signal. The filter takes the modulus of the complex output, as in lecture 13.

In [ ]:
# the whole output of the filter, with the filter of pycbc and the same crop as in analyze
phi_right = pycbc_matched_filter(template, injected, psd=psd, low_frequency_cutoff=F_LOW)
phi_right = phi_right.crop(6, 4)
t_phi = phi_right.sample_times.numpy() - T_DEMO
abs_phi = np.abs(np.asarray(phi_right))

# the output for the template at two positions: one second before the signal and on it
for shift in (-1.0, 0.0):
    k = np.argmin(np.abs(t_phi - shift))
    print(f"template at {shift:+.1f} s: matched_filter {abs_phi[k]:4.1f}")

**What the result shows.** The output stays at the noise level while the template lies on noise. One second before the signal, $\Phi$ is 0.8. On the signal it is 19.3, close to the $\varrho = 20$ we put in. The peak is a few milliseconds wide: the template has to be at the right time to better than that.

### The animation

The top panel shows the band-passed data in gray and the band-passed template in blue, placed at the arrival time $t_{c}$ of the bottom axis. The template is drawn at the size of the signal. The bottom panel shows $\Phi$ up to that time. The dashed line is the threshold $\Phi_{*} = 8$. The template moves slowly near the signal, so that the growth of the peak is visible.

In [ ]:
def shift_schedule(first, last, n_far=35, n_near=45, half_width=0.12):
    """Choose the positions of the template: fast far from the signal, slow near it.

    Args:
        first, last: the first and the last position (s, from the injection).
        n_far: the number of positions before the signal (default 35).
        n_near: the number of positions within half_width of the signal (default 45).
        half_width: the half width of the slow stretch (s, default 0.12).

    Returns:
        shifts: the positions of the template, an array in seconds.
    """
    # approach, stay near the signal, leave
    approach = np.linspace(first, -half_width, n_far, endpoint=False)
    on_signal = np.linspace(-half_width, half_width, n_near, endpoint=False)
    leave = np.linspace(half_width, last, n_far // 3)
    shifts = np.concatenate([approach, on_signal, leave])
    return shifts


def make_scene(top, bottom, template_drawn, name, color, curve, shifts, x_range,
               threshold=8.0, first=True):
    """Draw one panel pair of the animation, without the moving parts.

    The top axes have the band-passed data (b_injected) and the template. The bottom
    axes have the output of the filter for this template.

    Args:
        top, bottom: the two axes of the panel.
        template_drawn: the band-passed template to draw, at the size of the signal.
        name: the masses of the template, as text.
        color: the color of the template and of its output.
        curve: the pair (times, |Phi|) of the output of this template.
        shifts: the positions of the template (s).
        x_range: the limits of the time axis (s).
        threshold: the threshold Phi_* drawn as a dashed line (default 8).
        first: True for the panel at the left, which has the axis labels (default True).

    Returns:
        scene: a dictionary with the moving parts and what update_scene needs.
    """
    # the band-passed data, which do not move
    window = (t_rel >= x_range[0]) & (t_rel <= x_range[1])
    x_data = t_rel[window]
    y_data = b_injected[window] / noise_rms
    top.plot(x_data, y_data, color=P['GREY_MID'], lw=0.5, label='band-passed data')
    top_limit = 1.5 * np.abs(y_data).max()      # room for the legend above the data
    top.set_ylim(-top_limit, top_limit)

    # the template and the output are empty: update_scene fills them
    template_line, = top.plot([], [], color=color, lw=1.2, label='template')
    output_line, = bottom.plot([], [], color=color, lw=1.4, label='$\\Phi$')
    dot, = bottom.plot([], [], 'o', color=color, ms=5)

    # the threshold, and the final peak of this output in the label of the panel
    bottom.axhline(threshold, color=P['GREY_DARK'], lw=1.0, ls='--', label=f'$\\Phi_{{*}} = {threshold:g}$')
    x_out, y_out = curve
    scanned = (x_out >= shifts[0] - data.delta_t) & (x_out <= shifts[-1])
    final_peak = y_out[scanned].max()
    final_time = x_out[scanned][np.argmax(y_out[scanned])]
    top.set_title(f'{name} $M_{{\\odot}}$,  peak $\\Phi$ = {final_peak:.1f} at {1e3 * final_time:+.1f} ms',
                  loc='left')
    # the true arrival time of the signal
    bottom.axvline(0.0, color=P['GREY_DARK'], lw=1.0, ls=':')
    bottom.set_xlim(*x_range)

    # the labels, and the legends in the empty corners, away from the curves
    if first:
        top.set_ylabel('band-passed strain\n[rms of the noise]')
        bottom.set_ylabel('$\\Phi$')
    top.legend(loc='upper left', ncol=2, frameon=False, fontsize=9)
    bottom.legend(loc='upper left', ncol=2, frameon=False, fontsize=9)
    readout = bottom.text(0.04, 0.70, '', transform=bottom.transAxes, color=color)

    scene = dict(template=template_drawn, curve=curve, window=window, x_data=x_data,
                 template_line=template_line, output_line=output_line, dot=dot,
                 readout=readout, x0=shifts[0], dt=data.delta_t)
    return scene


def make_scenes(drawn, names, colors, curves, shifts, x_range, width=8.6):
    """Make a figure with one panel pair for each template, all with the same axes.

    Args:
        drawn: the band-passed templates to draw, one for each panel.
        names: the masses of the templates, as text.
        colors: the colors of the templates.
        curves: for each template, the pair (times, |Phi|) of its output.
        shifts: the positions of the template (s).
        x_range: the limits of the time axis (s).
        width: the width of the figure (inches, default 8.6).

    Returns:
        fig: the figure.
        scenes: the list of the scenes, one for each panel pair.
        title: the title of the figure, which update_scene changes.
    """
    # the same time axis and the same vertical axes in all the panels
    n = len(names)
    fig, axes = plt.subplots(2, n, figsize=(width, 5.2), sharex=True, sharey='row',
                             squeeze=False, dpi=72)
    fig.subplots_adjust(left=0.85 / width, right=1 - 0.15 / width, top=0.87,
                        bottom=0.13, hspace=0.12, wspace=0.05)

    # one scene for each template
    scenes = []
    for rank in range(n):
        scene = make_scene(axes[0][rank], axes[1][rank], drawn[rank], names[rank],
                           colors[rank], curves[rank], shifts, x_range, first=(rank == 0))
        scenes.append(scene)

    # the same limits for the output in all the panels
    highest = 0.0
    for times_c, values_c in curves:
        inside = (times_c >= x_range[0]) & (times_c <= x_range[1])
        highest = max(highest, values_c[inside].max())
    axes[1][0].set_ylim(0, 1.15 * highest)
    fig.supxlabel('arrival time $t_{c}$ of the template, from the injection [s]')
    title = fig.suptitle('', x=0.5, y=0.985, fontsize=11)
    return fig, scenes, title


def update_scene(frame, scenes, title, shifts):
    """Move the template of every panel to the position of one frame and draw each output up to it.

    Args:
        frame: the number of the frame.
        scenes: the list of the scenes made by make_scenes.
        title: the title of the figure.
        shifts: the positions of the template (s).

    Returns:
        Nothing: the lines of the figure are changed in place.
    """
    # the position of the template in this frame, in samples
    shift = shifts[frame]
    step = int(round(shift / scenes[0]['dt']))
    for scene in scenes:
        moved = np.roll(scene['template'], step)[scene['window']] / noise_rms
        scene['template_line'].set_data(scene['x_data'], moved)
        # the output from the first position of the template up to this one
        x, y = scene['curve']
        seen = (x >= scene['x0'] - scene['dt']) & (x <= shift)
        scene['output_line'].set_data(x[seen], y[seen])
        scene['dot'].set_data([x[seen][-1]], [y[seen][-1]])
        scene['readout'].set_text(f"$\\Phi$ = {y[seen][-1]:.1f}")
    title.set_text(f"template at {shift:+.2f} s")


def make_animation(fig, scenes, title, shifts, fps=20):
    """Turn the scenes into an animation that runs in the page.

    Args:
        fig: the figure made by make_scenes.
        scenes: the list of the scenes.
        title: the title of the figure.
        shifts: the positions of the template (s).
        fps: the frames per second (default 20).

    Returns:
        html: the animation as HTML, with small frames so that the page stays light.
    """
    anim = FuncAnimation(fig, update_scene, frames=len(shifts),
                         fargs=(scenes, title, shifts), interval=50)
    plt.close(fig)
    with plt.rc_context({'savefig.dpi': 72, 'animation.frame_format': 'jpeg'}):
        html = anim.to_jshtml(fps=fps)
    return html


# the right template: band-passed at the size of the signal, and its output
drawn_right = bandpass_series(chirp_in_strain(36, 29, T_DEMO, rho=RHO_TARGET))
shifts = shift_schedule(-0.95, 0.25)
fig, scenes, title = make_scenes([drawn_right], ['36 + 29'], [P['BLUE']],
                                 [(t_phi, abs_phi)], shifts, (-1.1, 0.35))
html = make_animation(fig, scenes, title, shifts)
print(f"{len(shifts)} frames, {len(shifts) / 20:.1f} s, {len(html) / 1e6:.1f} MB of HTML")
HTML(html)

In the animation $\Phi$ stays at the noise level while the template is on noise and rises when the template is on the signal.

### Right and wrong parameters

The result depends on the template. Lecture 13 gives the number: on a signal the mean of $\Phi$ is $\mathcal{O}\varrho$, the overlap of the template with the signal times the SNR of the signal. We keep the data and try two templates with wrong masses. The 30 + 30 template is close to the right one. The 20 + 15 template is far from it. For each template we find the peak of $\Phi$ and its time. Then we compute $\mathcal{O}$, the overlap, from the two templates alone with `match` of `pycbc`, the name that the LVK papers give to $\mathcal{O}$. It is the normalized product at its maximum over time and phase. The mismatch is $\mu = 1-\mathcal{O}$. According to lecture 13 the ratio of the peaks is $\mathcal{O}$ up to the noise.

In [ ]:
# The right template and two wrong ones, on the same data
masses = [(36, 29), (30, 30), (20, 15)]
templates = {}
for m in masses:
    if m == (36, 29):
        templates[m] = template
    else:
        templates[m] = make_template(m[0], m[1])

# the output of each template, its peak near the injection and the overlap with the right one
outputs = {}
peak = {}
peak_time = {}
print(f"{'template':>9} {'peak':>6} {'time':>9} {'ratio':>6} {'overlap':>8} {'mismatch':>9}")
for m in masses:
    phi = pycbc_matched_filter(templates[m], injected, psd=psd, low_frequency_cutoff=F_LOW)
    phi = phi.crop(6, 4)
    outputs[m] = np.abs(np.asarray(phi))
    # the peak within 0.3 s of the injection
    near = np.abs(t_phi) < 0.3
    k = np.arange(len(t_phi))[near][np.argmax(outputs[m][near])]
    peak[m] = outputs[m][k]
    peak_time[m] = t_phi[k]
    # the overlap of the template with the right one: maximum over time and phase
    overlap, _ = match(templates[(36, 29)], templates[m], psd=psd, low_frequency_cutoff=F_LOW)
    print(f"{m[0]:4d} +{m[1]:3d} {peak[m]:6.1f} {1e3 * t_phi[k]:6.1f} ms "
          f"{peak[m] / peak[(36, 29)]:6.2f} {overlap:8.2f} {1 - overlap:9.2f}")

# a zoom on the peaks: the three outputs within 15 ms of the arrival time of the signal
zoom = np.abs(t_phi) < 0.015
colors_rw = {(36, 29): P['BLUE'], (30, 30): P['SAGE'], (20, 15): P['BRICK']}
fig, ax = plt.subplots(figsize=(9, 4))
for m in masses:
    label = f"{m[0]} + {m[1]}: peak {peak[m]:.1f} at {1e3 * peak_time[m]:+.1f} ms"
    ax.plot(1e3 * t_phi[zoom], outputs[m][zoom], color=colors_rw[m], lw=1.5, label=label)
    ax.plot([1e3 * peak_time[m]], [peak[m]], 'o', color=colors_rw[m], ms=7)
# the true arrival time of the signal
ax.axvline(0.0, color=P['GREY_DARK'], lw=1.2, ls=':', label='true arrival time')
ax.set_xlabel('arrival time $t_c$ of the template, from the injection [ms]')
ax.set_ylabel(r'$\Phi$')
ax.set_ylim(0, 1.45 * peak[(36, 29)])
ax.legend(frameon=False, loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()


**What the result shows.** The ratio of the peaks is the overlap. The 30 + 30 template gives a peak of 17.0 and a ratio of 0.88. The overlap from the two templates alone is $\mathcal{O} = 0.88$. The 20 + 15 template gives 8.9, a ratio of 0.46 against $\mathcal{O} = 0.44$. The difference of 0.02 is the noise of one realization. The mismatch $\mu$ is 0.12 and 0.56.

A wrong template gives two errors: a lower $\Phi$ and a wrong arrival time. The zoom shows the second. The right template peaks at the true arrival time. The 30 + 30 template peaks 1.5 ms late and the 20 + 15 template 5.4 ms late: a wrong chirp rate moves the best alignment. On the scale of the animation, almost one second, these shifts are too small to see. At $\varrho = 20$ even the 20 + 15 template stays above the threshold $\Phi_{*} = 8$. A signal with $\varrho = 12$ would give $0.44\times12 = 5.3$ and would not be detected. A bank of lecture 13 places its templates close enough that no signal loses more than the fraction $1-\mathrm{MM}$ of its $\Phi$, with MM the minimal match.

The same data and the three templates, now in three panels. Each panel has its own template, slid along the same band-passed data. Below it is the output $\Phi(t_{c})$ of that template. The vertical axes are the same in the three panels and so is the threshold $\Phi_{*}$. The label of each panel gives the masses, the peak of $\Phi$ and its time from the injection. The dotted line is the true arrival time of the signal.

In [ ]:
# The same animation in three panels, one for each template
names = []
curves = []
drawn = []
for m in masses:
    names.append(f'{m[0]} + {m[1]}')
    curves.append((t_phi, outputs[m]))
    # each template is band-passed at the size of the signal
    chirp = chirp_in_strain(m[0], m[1], T_DEMO, rho=RHO_TARGET)
    drawn.append(bandpass_series(chirp))
colors = [P['BLUE'], P['SAGE'], P['BRICK']]

shifts = shift_schedule(-0.55, 0.2, n_far=14, n_near=30, half_width=0.06)
fig, scenes, title = make_scenes(drawn, names, colors, curves, shifts, (-0.6, 0.25), width=12.6)
html = make_animation(fig, scenes, title, shifts)
print(f"{len(shifts)} frames, {len(shifts) / 20:.1f} s, {len(html) / 1e6:.1f} MB of HTML")
HTML(html)

**What the result shows.** In the three panels the output stays at the noise level until the template reaches the signal. The 36 + 29 template reaches 19.3 and the 30 + 30 template 17.0: both are above the threshold $\Phi_{*} = 8$. The 20 + 15 template reaches 8.9, just above it. The peaks are at 0.0, +1.5 and +5.4 ms from the injection, as the labels say: on this scale the shift is too small to see, which is why the zoom above is drawn. The fractions of the right peak are the ratios of the table above.

## A3. The background: the output on noise alone

We run the filter over the whole record with nothing added and compute the distribution of $|\Phi|$.

We plot a histogram against the Rayleigh distribution that unit variance noise gives and report the largest value away from GW150914. For noise alone the Rayleigh tail is $F = \mathrm{e}^{-\Phi_{*}^{2}/2}$ for one arrival time and one template. The loudest of $N_{\mathrm{trials}}$ trials, here the samples of the output, is then expected near $\Phi_{*} = \sqrt{2\ln N_{\mathrm{trials}}}$. We compare the two values.

In [ ]:
def find_peak(phi, t0=None, half_width=0.3, skip=None):
    """Find the largest |Phi|, near a time or in the whole output.

    Args:
        phi: the output of the filter, a pycbc TimeSeries.
        t0: the time where the peak is expected (GPS s, default None: anywhere).
        half_width: the half width of the search around t0 (s, default 0.3).
        skip: a pair of times (GPS s) between which the output is not searched
            (default None: nothing is skipped).

    Returns:
        k: the index of the peak in phi.
    """
    # the times where the peak is searched
    t_axis = phi.sample_times.numpy()
    allowed = np.ones(len(t_axis), dtype=bool)
    if t0 is not None:
        allowed = allowed & (np.abs(t_axis - t0) < half_width)
    if skip is not None:
        allowed = allowed & ((t_axis < skip[0]) | (t_axis > skip[1]))
    # the largest |Phi| among them
    indices = np.arange(len(t_axis))[allowed]
    k = indices[np.argmax(np.abs(np.asarray(phi))[allowed])]
    return k


# the filter over the whole record with nothing added
phi = analyze(data, template, psd)
abs_phi = np.abs(np.asarray(phi))
t_axis = phi.sample_times.numpy()

# the output away from the event
quiet = abs_phi[np.abs(t_axis - GPS_EVENT) > 0.5]
print(f"{t_axis[-1] - t_axis[0]:.0f} s of filter output, "
      f"{len(quiet) * phi.delta_t:.0f} s of it away from the event")

# the histogram of |Phi| against the Rayleigh distribution
grid = np.linspace(0, 6, 300)
plt.figure(figsize=(7.2, 4.2))
plt.hist(quiet, bins=80, density=True, color=P['BLUE_LIGHT'],
         edgecolor=P['BLUE'], lw=0.5, label='the record, away from the event')
plt.plot(grid, grid * np.exp(-grid ** 2 / 2), color=P['BRICK'], lw=2.0,
         label='Rayleigh, unit variance noise')
plt.yscale('log')
plt.ylim(1e-4, 2)
plt.xlabel(r'$|\Phi|$')
plt.ylabel('density')
plt.legend(frameon=False)
plt.show()

# the mean, the spread and the loudest value, against the theory
print(f"mean {quiet.mean():.2f} against 1.25, "
      f"standard deviation {quiet.std():.2f} against 0.65")
print(f"loudest away from the event: {quiet.max():.1f}")
print(f"Rayleigh: the loudest of N_trials = {len(quiet)} samples is expected near "
      f"Phi_* = {np.sqrt(2 * np.log(len(quiet))):.1f}")
print(f"at the event: {abs_phi[np.abs(t_axis - GPS_EVENT) < 0.1].max():.1f}")

**What the result shows.** The bulk follows the Rayleigh curve, which is the check that the noise spectrum was estimated well. The event is at about 16.5 and the loudest thing that is not the event is 4.7, in **seventeen seconds** of data. The Rayleigh tail expects 4.6 for the loudest of that many samples, so the tail agrees with the theory. The samples are correlated, so this is a rough guide.

A search runs for months and has about $7\times10^{13}$ trials in a day. Its threshold $\Phi_{*}$ has to be far above 4.7: $\sqrt{2\ln N_{\mathrm{trials}}} \approx 8$ (lecture 13). The false alarm rate of a given threshold is measured on the data with time slides, as in lecture 13.

### With a signal in the data

We run the same filter on the data of A2, with the 36 + 29 signal of SNR 20, and draw the same histogram of $|\Phi|$ against the Rayleigh distribution, with the peak of the signal as one line.


In [ ]:
# the filter on the data with the signal of A2 (SNR 20), with the same crop
phi_with = analyze(injected, template, psd)
abs_with = np.abs(np.asarray(phi_with))
t_with = phi_with.sample_times.numpy()

# the noise: away from the real event and more than 10 ms from the signal
noise_only = (np.abs(t_with - GPS_EVENT) > 0.5) & (np.abs(t_with - T_DEMO) > 0.010)
# the signal: the largest |Phi| within 10 ms of its arrival time
near_signal = np.abs(t_with - T_DEMO) <= 0.010
peak_signal = abs_with[near_signal].max()

# the histogram of the noise against the Rayleigh distribution, and the peak of the signal
grid = np.linspace(0, 22, 400)
plt.figure(figsize=(7.2, 4.2))
plt.hist(abs_with[noise_only], bins=np.arange(0, 22, 0.25), density=True, color=P['BLUE_LIGHT'],
         edgecolor=P['BLUE'], lw=0.5, label='the record, away from the signal')
plt.plot(grid, grid * np.exp(-grid ** 2 / 2), color=P['BRICK'], lw=2.0,
         label='Rayleigh, unit variance noise')
plt.axvline(peak_signal, color=P['SAGE_DARK'], lw=2.0, label='the peak of the signal')
plt.yscale('log')
plt.ylim(1e-4, 2)
plt.xlabel(r'$|\Phi|$')
plt.ylabel('density')
plt.legend(frameon=False, loc='upper center', handlelength=2.5)
plt.show()

# the peak of the signal, and the largest value of the noise
print(f"the peak of the signal: {peak_signal:.1f}; the largest value of the noise: {abs_with[noise_only].max():.1f}")


**What the result shows.** The noise follows the Rayleigh curve and stays below 4.7. The signal is one peak, at 19.3, far outside the statistics of the noise.

$\Phi$ is computed at every sample, every 0.49 ms. Near the arrival time it rises and falls over a few milliseconds, as in the animation of A2. The samples on the sides of the peak belong to the same signal. A histogram counts samples, so the signal is drawn as one line at its peak.


## A4. A simulated glitch

A transient of the instrument is modeled by a sine-Gaussian

$$
g(t) = A \exp\!\left[-\left(\frac{t-t_{0}}{\tau}\right)^{2}\right]
\cos\!\left[2\pi f_{0}(t-t_{0})\right],
\qquad \tau = \frac{Q}{2\pi f_{0}}
$$

We inject two of them into a quiet stretch: a **blip**, $f_{0} = 150$ Hz and $Q = 3$, a few milliseconds long and broad in frequency; and a narrow band burst at the same frequency with $Q = 40$.

We plot both in time first, then run the matched filter on them and report $|\Phi|$.

In [ ]:
t_data = np.arange(len(data)) * data.delta_t + float(data.start_time)
T_GLITCH = float(data.start_time) + 12.0


def sine_gaussian(t_arr, t0, f0=150.0, q=3.0, amplitude=3e-21):
    """Compute a sine-Gaussian, the model of a glitch, with tau = Q / (2 pi f0).

    Args:
        t_arr: the times (s).
        t0: the central time (s).
        f0: the central frequency (Hz, default 150).
        q: the quality factor (default 3).
        amplitude: the amplitude (strain, default 3e-21).

    Returns:
        waveform: the sine-Gaussian at t_arr.
    """
    tau = q / (2 * np.pi * f0)
    waveform = amplitude * np.exp(-((t_arr - t0) / tau) ** 2) \
            * np.cos(2 * np.pi * f0 * (t_arr - t0))
    return waveform


# the two glitches: a blip and a narrow band burst, at the same frequency
glitches = {'blip, Q = 3': sine_gaussian(t_data, T_GLITCH, f0=150.0, q=3.0, amplitude=3e-21),
            'narrow band, Q = 40': sine_gaussian(t_data, T_GLITCH, f0=150.0, q=40.0, amplitude=1.2e-21)}

# the two glitches in time, against the rms of the data (the gray band)
rms = np.std(np.asarray(data))
near_glitch = np.abs(t_data - T_GLITCH) < 0.15
plt.figure(figsize=(10, 3.2))
k_panel = 1
for name, g in glitches.items():
    plt.subplot(1, 2, k_panel)
    plt.axhspan(-rms, rms, color=P['GREY_MID'], alpha=0.25)
    plt.plot(t_data[near_glitch] - T_GLITCH, g[near_glitch], color=P['BRICK'], lw=1.2)
    plt.title(name, loc='left')
    plt.xlabel('time from the glitch [s]')
    if k_panel == 1:
        plt.ylabel('strain')
    k_panel = k_panel + 1
plt.show()

# then the matched filter on each
for name, g in glitches.items():
    series = TimeSeries(np.asarray(data) + g, delta_t=data.delta_t, epoch=data.start_time)
    phi = analyze(series, template, psd)
    k = find_peak(phi, T_GLITCH, half_width=0.2)
    print(f"{name:22s} Phi = {abs(phi[k]):5.1f}")

# and the same number for the real event
phi = analyze(data, template, psd)
k = find_peak(phi, GPS_EVENT, half_width=0.1)
print(f"{EVENT:22s} Phi = {abs(phi[k]):5.1f}")

**What the result shows.** In time the blip lasts a few milliseconds and the narrow band burst several tens of them. Neither rises above the rms of the data, the gray band. The blip gives about 15 on the filter and the narrow band burst about 12, against 16.5 for the real event. **On $\Phi$ alone the glitches cannot be separated from the event.** A search with the threshold $\Phi_{*} \approx 8$ on $\Phi$ would report all three as triggers.

A glitch gives a large $\Phi$ with a template that does not look like it. In one detector, $\Phi$ alone does not tell a glitch from a signal. A real search asks for the same template in a second detector, within the light travel time: this is the coincidence of A5. Searches also check the quality of the data and some pipelines add a test of the shape of the trigger (lecture 13, auxiliary slides).

## A5. Coincidence: the same template in two detectors

This section shows the coincidence of lecture 13. A wave reaches Hanford and Livingston within the light travel time between the sites, 10 ms. A glitch is in one detector only.

We load the Livingston data with `load_raw` and `condition` of A1. We run the same 36 + 29 template on both records. Here the real event is the subject. The function `coincident_peaks` finds the peak of $|\Phi|$ in each detector near a time and the delay between the two peaks. The function `plot_two_detectors` draws the two outputs on the same time axis, with the threshold $\Phi_{*} = 8$ and the window of $\pm 10$ ms around the Hanford peak.

In [ ]:
LIGHT_TIME = 0.010       # light travel time between the sites [s]


def coincident_peaks(phi_h1, phi_l1, t0, half_width=0.1, window=LIGHT_TIME, threshold=8.0):
    """Find the peak of |Phi| in each detector near t0 and check the coincidence.

    Args:
        phi_h1, phi_l1: the outputs of the filter in H1 and in L1, pycbc TimeSeries.
        t0: the time where the peaks are searched (GPS s).
        half_width: the half width of the search around t0 (s, default 0.1).
        window: the largest delay of a coincidence (s, default 0.010).
        threshold: the threshold Phi_* (default 8).

    Returns:
        peaks: a dictionary with the two peaks, their times (GPS s), the delay
            t_H1 - t_L1 (s) and whether they are a coincidence.
    """
    # the peak in each detector, searched in the same stretch of time
    k_h1 = find_peak(phi_h1, t0, half_width)
    k_l1 = find_peak(phi_l1, t0, half_width)
    peaks = {'H1': abs(phi_h1[k_h1]), 'L1': abs(phi_l1[k_l1]),
             't_H1': float(phi_h1.sample_times[k_h1]), 't_L1': float(phi_l1.sample_times[k_l1])}
    peaks['delay'] = peaks['t_H1'] - peaks['t_L1']
    # a coincidence: both above the threshold and the delay inside the window
    above = (peaks['H1'] > threshold) and (peaks['L1'] > threshold)
    peaks['coincident'] = above and (abs(peaks['delay']) <= window)
    return peaks

In [ ]:
def plot_two_detectors(phi_h1, phi_l1, t_ref, peaks, window=LIGHT_TIME,
                       threshold=8.0, half_width=0.05):
    """Draw |Phi| of H1 and L1 around a time, with the threshold and the light travel window.

    The dots: the H1 peak and the largest |Phi| of L1 within the window around it.

    Args:
        phi_h1, phi_l1: the outputs of the filter in H1 and in L1, pycbc TimeSeries.
        t_ref: the time at the center of the plot (GPS s).
        peaks: the dictionary returned by coincident_peaks.
        window: the half width of the shaded window around the H1 peak (s, default 0.010).
        threshold: the threshold Phi_* (default 8).
        half_width: the half width of the plot (s, default 0.05).

    Returns:
        Nothing: the figure is shown.
    """
    plt.figure(figsize=(9, 3.8))
    # the two outputs on the same time axis, in ms from t_ref
    for phi, name, color in ((phi_h1, 'H1', P['BLUE']), (phi_l1, 'L1', P['SAGE'])):
        t_ms = 1e3 * (phi.sample_times.numpy() - t_ref)
        near = np.abs(t_ms) < 1e3 * half_width
        plt.plot(t_ms[near], np.abs(np.asarray(phi))[near], color=color, lw=1.3, label=name)
    # the H1 peak, and the largest |Phi| of L1 within the window around it
    k_l1 = find_peak(phi_l1, peaks['t_H1'], half_width=window)
    t_l1 = 1e3 * (float(phi_l1.sample_times[k_l1]) - t_ref)
    plt.plot([1e3 * (peaks['t_H1'] - t_ref)], [peaks['H1']], 'o', color=P['BLUE'], ms=6)
    plt.plot([t_l1], [abs(phi_l1[k_l1])], 'o', color=P['SAGE'], ms=6)
    # the window of the light travel time around the H1 peak, and the threshold
    t_h1 = 1e3 * (peaks['t_H1'] - t_ref)
    plt.axvspan(t_h1 - 1e3 * window, t_h1 + 1e3 * window, color=P['GREY_MID'], alpha=0.25,
                label='light travel window')
    plt.axhline(threshold, color=P['GREY_DARK'], lw=1.0, ls='--', label=r'$\Phi_{*}$')
    plt.xlabel(f'time from GPS {t_ref:.2f} [ms]')
    plt.ylabel(r'$|\Phi|$')
    plt.ylim(0, 1.15 * max(peaks['H1'], peaks['L1'], threshold))
    plt.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=4, frameon=False)
    plt.show()

In [ ]:
# the Livingston data and their noise spectrum, conditioned as in A1
data_l1, psd_l1 = condition(load_raw('L1'))

# the same template on both records, with the noise spectrum of each detector
phi_h1 = analyze(data, template, psd)
phi_l1 = analyze(data_l1, template, psd_l1)

# the two peaks near the event and their delay
event = coincident_peaks(phi_h1, phi_l1, GPS_EVENT)
print(f"H1: Phi = {event['H1']:.1f} at GPS {event['t_H1']:.4f}")
print(f"L1: Phi = {event['L1']:.1f} at GPS {event['t_L1']:.4f}")
print(f"delay t_H1 - t_L1 = {1e3 * event['delay']:+.1f} ms, coincidence: {event['coincident']}")
plot_two_detectors(phi_h1, phi_l1, GPS_EVENT, event)

**What the result shows.** Both detectors give a peak above the threshold with the same template: 16.5 in Hanford and 13.0 in Livingston. The wave reaches Livingston first. The Hanford peak is 7.3 ms later, inside the window of 10 ms. The two triggers are a coincidence.

The noises of the two detectors are independent, so the squares of the two outputs add. This gives the statistic of the network of lecture 13.

In [ ]:
# the statistic of the network: the squares of the outputs of the two detectors add
phi_net = np.sqrt(event['H1'] ** 2 + event['L1'] ** 2)
print(f"Phi_net = sqrt({event['H1']:.1f}^2 + {event['L1']:.1f}^2) = {phi_net:.1f}")

**What the result shows.** $\Phi_{\mathrm{net}} = \sqrt{16.5^{2} + 13.0^{2}} = 21.0$. Two detectors give a larger statistic than either one alone.

### A glitch is in one detector

We take the blip of A4 in the Hanford data and run the same template on Livingston at the same time. Nothing is added to Livingston.

In [ ]:
# the blip of A4 in the Hanford data; Livingston as it is
blip_h1 = sine_gaussian(t_data, T_GLITCH, f0=150.0, q=3.0, amplitude=3e-21)
series_h1 = TimeSeries(np.asarray(data) + blip_h1, delta_t=data.delta_t, epoch=data.start_time)
phi_glitch = analyze(series_h1, template, psd)

# the two peaks near the glitch, and the largest |Phi| of L1 within 10 ms of the H1 peak
glitch = coincident_peaks(phi_glitch, phi_l1, T_GLITCH)
k_window = find_peak(phi_l1, glitch['t_H1'], half_width=LIGHT_TIME)
print(f"H1: Phi = {glitch['H1']:.1f}")
print(f"L1: largest Phi within 10 ms of the H1 peak = {abs(phi_l1[k_window]):.1f}")
print(f"coincidence: {glitch['coincident']}")
plot_two_detectors(phi_glitch, phi_l1, T_GLITCH, glitch)

**What the result shows.** The blip gives 14.8 in Hanford, above the threshold, as in A4. Livingston has only noise at that time: its largest $\Phi$ within 10 ms of the Hanford peak is 2.4. The event is a coincidence and the glitch is not. **Coincidence between detectors is what separates a signal from a glitch in a search.**

A coincidence can also happen by chance, between two noise triggers in the two detectors. How often it happens is measured by shifting one detector in time: the time slides of lecture 13. One record of 32 s cannot show it.

## A6. A claim to test

Which template do the glitches affect most? We repeat A4 with a template for a lighter binary of 10 + 10 solar masses. This template lasts about six seconds in band and not a fraction of a second. We find whether $\Phi$ of the blip rises or falls.

In [ ]:
# A 10 + 10 template lasts about 6 s in band from 20 Hz (3 s from the 25 Hz used here),
# against 0.7 s for 36 + 29
light_template = make_template(10, 10, f_low=25)

# the blip of A4 added to the data
blip = sine_gaussian(t_data, T_GLITCH, f0=150.0, q=3.0, amplitude=3e-21)
series = TimeSeries(np.asarray(data) + blip, delta_t=data.delta_t, epoch=data.start_time)

# Phi of the blip with each template
for name, tmpl in (('36 + 29', template), ('10 + 10', light_template)):
    phi = analyze(series, tmpl, psd)
    k = find_peak(phi, T_GLITCH, half_width=0.3)
    print(f"{name:8s} Phi = {abs(phi[k]):5.1f}")

**What the result shows.** The light template picks up **less** $\Phi$ from the blip, about 9 against 15: a short broadband transient overlaps a long template poorly.

In general, **a glitch affects a template when its duration and bandwidth resemble those of the template.** Blips affect the high mass, short templates. Slow wandering noise affects the low mass, long templates. A search over a wide bank has both cases.

## Part B. Your turn: inject a signal and search for it

Part A showed the filter at work. In Part B we choose a signal and the way to search for it. The five functions below are defined for you. They call the functions of Part A: `load_raw`, `condition`, `make_template`, `inject`, `analyze`, `find_peak` and `bandpass_series`.

- `load_data(detector='H1', f_high_pass=15.0, fs_out=2048.0, plot=False)` loads the strain of one detector around GW150914, high-passes it, resamples it, crops the edges and estimates the PSD. With `plot=True` it draws the strain and the spectrum. It returns `data, psd`.
- `make_template(mass1, mass2, f_low=20.0, approximant='IMRPhenomD', distance=1.0)` builds the waveform of a binary with these masses, for a source at `distance` in Mpc, as long as the data and with the merger at time zero. `approximant` is the waveform model. Models that work here: `IMRPhenomD`, `IMRPhenomXAS`, `SEOBNRv4_opt` and `TaylorT4`.
- `inject_signal(data, mass1, mass2, distance, t_merger=None, f_low=20.0, approximant='IMRPhenomD')` adds the signal of a binary with these masses at this distance (Mpc) to the data. The merger is at `t_merger`, by default 8 s after the start of the data. It prints what was injected and returns the new data.
- `matched_filter(data, template, psd, f_low=20.0, skip=EVENT_SKIP)` searches the data with one template. It returns `phi, peak, t_peak`: the output $\Phi(t_{c})$, its largest value and the time of that value. The real event GW150914 is skipped by default.
- `plot_results(data, template, psd, phi, t_peak, t_inject=None, threshold=8.0)` draws two panels. The first has the band-passed data around the peak with the template on them. The second has $\Phi(t_{c})$ over the record with the threshold $\Phi_{*}$.

**The signal that is injected.** It is the waveform of a compact binary coalescence: two bodies on a quasi-circular orbit, with no spins. The source is face-on and overhead, so the detector sees the plus polarization with response 1. This is the optimal orientation. The waveform at distance $d$ is the waveform at 1 Mpc divided by $d$, so the SNR at distance $d$ is the SNR at 1 Mpc divided by $d$. Left out: the spins, the eccentricity, the antenna pattern, the sky position, the redshift and the second detector. The signal is added to the data of H1 and not to a simulated noise.

**What to choose.** Masses between about 10 and 80 solar masses and distances between about 100 Mpc and a few Gpc all work. At 100 Mpc the SNR is 60 for 10 + 10 and 295 for 80 + 80. At 1 Gpc it is 6 and 30. The SNR is 8, the threshold $\Phi_{*}$, at 745 Mpc for 10 + 10, at 1.9 Gpc for 36 + 29 and at 3.7 Gpc for 80 + 80. A signal that is too far gives a peak that is not above the threshold. This is a valid result: the report says that the data do not allow a claim.

In [ ]:
# the stretch of the record with the real event: the search skips it (GPS s)
EVENT_SKIP = (GPS_EVENT - 0.5, GPS_EVENT + 0.5)


def load_data(detector='H1', f_high_pass=15.0, fs_out=2048.0, plot=False):
    """Load the strain of one detector around the event, condition it and, if asked, plot it.

    Args:
        detector: 'H1' for Hanford or 'L1' for Livingston (default 'H1').
        f_high_pass: the frequency below which the strain is removed (Hz, default 15).
        fs_out: the sampling frequency after the resampling (Hz, default 2048).
        plot: True to draw the strain and the amplitude spectral density (default False).

    Returns:
        data: the conditioned strain, a pycbc TimeSeries of 28 s.
        psd: the one-sided PSD S_n(f) on the frequency grid of the data.
    """
    # the strain of the file, high-passed and resampled, then cropped and its PSD estimated
    raw_strain = load_raw(detector, f_high_pass, fs_out)
    conditioned, noise_psd = condition(raw_strain)
    # the strain in time and the amplitude spectral density
    if plot:
        t_axis = conditioned.sample_times.numpy() - float(conditioned.start_time)
        asd_freqs = noise_psd.sample_frequencies.numpy()
        asd = np.sqrt(np.asarray(noise_psd))
        plt.figure(figsize=(9, 3.6))
        plt.subplot(1, 2, 1)
        plt.plot(t_axis, np.asarray(conditioned), color=P['GREY_MID'], lw=0.4)
        plt.xlabel('time from the start of the record [s]')
        plt.ylabel('strain')
        plt.title(f'{detector}', loc='left')
        plt.subplot(1, 2, 2)
        inside = (asd_freqs >= 10) & (asd_freqs <= 1000)
        plt.loglog(asd_freqs[inside], asd[inside], color=P['BLUE'], lw=1.2)
        plt.xlabel('frequency [Hz]')
        plt.ylabel(r'ASD $\sqrt{S_n}$ [1/$\sqrt{\mathrm{Hz}}$]')
        plt.tight_layout()
        plt.show()
    return conditioned, noise_psd


def inject_signal(data, mass1, mass2, distance, t_merger=None, f_low=20.0,
                  approximant='IMRPhenomD'):
    """Add the signal of a binary to the data and print what was injected.

    The signal is a compact binary coalescence: a quasi-circular orbit, no spins,
    seen face-on and overhead, so the detector sees h_+ with response 1. The
    antenna pattern, the sky position and the redshift are left out.

    Args:
        data: the data, a pycbc TimeSeries.
        mass1, mass2: the masses of the binary (solar masses).
        distance: the distance of the source (Mpc).
        t_merger: the GPS time of the merger (s, default None: 8 s after the start
            of the data, away from the event).
        f_low: the lower edge of the band (Hz, default 20).
        approximant: the waveform model (default 'IMRPhenomD').

    Returns:
        with_signal: the data with the signal added.
    """
    # the merger 8 s after the start of the data, unless a time is given
    if t_merger is None:
        t_merger = float(data.start_time) + 8.0
    # the waveform at 1 Mpc is divided by the distance: the SNR falls as 1 / r
    scale = 1.0 / distance
    with_signal = inject(data, mass1, mass2, t_merger, scale, f_low, approximant)
    # what was injected
    print(f"injected: {mass1} + {mass2} solar masses at {distance} Mpc, "
          f"merger at GPS {t_merger:.2f} s, model {approximant}")
    print("          quasi-circular, non-spinning, face-on, no antenna pattern, no redshift")
    return with_signal


def matched_filter(data, template, psd, f_low=20.0, skip=EVENT_SKIP):
    """Search the data with one template: the output Phi(t_c) and its loudest peak.

    Args:
        data: the data, a pycbc TimeSeries.
        template: the template, as long as the data.
        psd: the one-sided PSD S_n(f) of the noise.
        f_low: the lower edge of the band (Hz, default 20).
        skip: the pair of times (GPS s) of the stretch that is not searched, the
            real event (default 0.5 s each side of the event; None searches all).

    Returns:
        phi: the complex output Phi(t_c) of the filter, a pycbc TimeSeries.
        peak: the largest |Phi| outside the skipped stretch.
        t_peak: the GPS time of the peak (s).
    """
    # the filter output of the whole record
    phi = analyze(data, template, psd, f_low=f_low)
    # the loudest peak outside the skipped stretch, with its time
    k = find_peak(phi, skip=skip)
    peak = abs(phi[k])
    t_peak = float(phi.sample_times[k])
    return phi, peak, t_peak


def plot_results(data, template, psd, phi, t_peak, t_inject=None, threshold=8.0,
                 skip=EVENT_SKIP, f_low=20.0, before=0.4, after=0.1):
    """Draw the band-passed data around the peak with the template, and Phi(t_c) with the threshold.

    Args:
        data: the data, a pycbc TimeSeries.
        template: the template, as long as the data.
        psd: the one-sided PSD S_n(f) of the noise.
        phi: the output Phi(t_c) returned by matched_filter.
        t_peak: the GPS time of the peak returned by matched_filter (s).
        t_inject: the GPS time of the injection (s, default None: not drawn).
        threshold: the threshold Phi_* (default 8).
        skip: the stretch that was not searched, shaded (default: around the event).
        f_low: the lower edge of the band (Hz, default 20).
        before, after: the time drawn before and after the peak (s, default 0.4 and 0.1).

    Returns:
        Nothing: the figure is shown.
    """
    # the band-passed data, in rms of the band-passed noise
    filtered = bandpass_series(data)
    noise = np.std(filtered)
    t_data = data.sample_times.numpy()
    # the best-fit template at the peak: the amplitude and the phase of Phi(t_peak)
    t_phi = phi.sample_times.numpy()
    k_phi = np.argmin(np.abs(t_phi - t_peak))
    value = np.asarray(phi)[k_phi]
    rho_template = sigma(template, psd=psd, low_frequency_cutoff=f_low)
    amplitude = np.abs(value) / rho_template
    rotated = (template.to_frequencyseries() * np.exp(1j * np.angle(value))).to_timeseries()
    # the band-passed template, with its merger at the time of the peak
    step = int(round((t_peak - float(data.start_time)) / data.delta_t))
    fitted = np.roll(bandpass_series(rotated), step) * amplitude / noise

    plt.figure(figsize=(9, 6))
    # the data around the peak with the template on them
    plt.subplot(2, 1, 1)
    near = (t_data > t_peak - before) & (t_data < t_peak + after)
    plt.plot(t_data[near] - t_peak, filtered[near] / noise, color=P['GREY_MID'], lw=0.6,
             label='band-passed data')
    plt.plot(t_data[near] - t_peak, fitted[near], color=P['BLUE'], lw=1.3,
             label='template at the peak')
    plt.xlabel('time from the peak [s]')
    plt.ylabel('band-passed strain\n[rms of the noise]')
    plt.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)

    # the output over the whole record, the threshold and the peak
    plt.subplot(2, 1, 2)
    start = float(data.start_time)
    plt.plot(t_phi - start, np.abs(np.asarray(phi)), color=P['BLUE'], lw=0.8, label=r'$|\Phi|$')
    plt.axhline(threshold, color=P['GREY_DARK'], lw=1.0, ls='--', label=f'$\\Phi_{{*}} = {threshold:g}$')
    plt.plot([t_peak - start], [np.abs(value)], 'o', color=P['BRICK'], ms=6, label='the peak')
    if t_inject is not None:
        plt.axvline(t_inject - start, color=P['SAGE'], lw=1.0, ls=':', label='the injection')
    if skip is not None:
        plt.axvspan(skip[0] - start, skip[1] - start, color=P['GREY_MID'], alpha=0.3,
                    label=f'{EVENT}, not searched')
    plt.xlabel('arrival time $t_{c}$ from the start of the record [s]')
    plt.ylabel(r'$\Phi$')
    plt.ylim(0, 1.15 * max(np.abs(value), threshold))
    plt.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=5, frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

We load the data of Hanford with the function `load_data` and draw them. They are the data of Part A: the strain with the event GW150914 and the noise spectrum.

In [ ]:
# the data and the noise spectrum of H1
data, psd = load_data('H1', plot=True)

### B1. Inject a signal

Choose the masses of a binary and its distance. Inject the signal in the data with `inject_signal`. Keep the result in a new variable: `data` stays as the data without the signal, which B4 needs.

Then find the SNR you expect, as in A1: build the template at your distance and compute $\sqrt{\langle h,h\rangle}$. Compute it for your masses and write the expected SNR for your distance.

*Hint:* `make_template(mass1, mass2, distance=d)` gives the template at the distance $d$ in Mpc and `sigma(template, psd=psd, low_frequency_cutoff=20)` gives the SNR you expect.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B2. Search with the same masses and with different masses

Search the data with the signal using `matched_filter`. Use first a template with the masses you injected. Then use a template with different masses: a few solar masses away and far away. For each template report the peak of $\Phi$ and its time.

Compare the peaks. By lecture 13 the mean of $\Phi$ on a signal is $\mathcal{O}\varrho$, with $\mathcal{O}$ the overlap of the template with the signal. A template with other masses has $\mathcal{O}<1$ and gives a lower peak. Draw the result for one template with `plot_results`.

*Hint:* `matched_filter(with_signal, template, psd)` returns `phi, peak, t_peak`. The time of the injection is `float(data.start_time) + 8`.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B3. Search with another waveform model

Search again with the masses you injected and another waveform model: change the argument `approximant` of `make_template`. The injection used `IMRPhenomD`. Try `IMRPhenomXAS`, `SEOBNRv4_opt` and `TaylorT4`. The first two contain the inspiral, the merger and the ringdown. `TaylorT4` is an inspiral only and stops before the merger.

Report the peak and its time for each model. Compare the peaks with the peak of the right template in B2. The difference is the loss that the waveform model alone gives, with the masses right.

*Hint:* `make_template(mass1, mass2, approximant='SEOBNRv4_opt')`.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B4. A claim of detection

Decide from your results whether the data allow a claim. Three questions:

1. Is the peak above the threshold $\Phi_{*} = 8$ of lecture 13?
2. Is it at the time of the injection?
3. How loud is the output on the same data without the injection? This is the background of this record.

State the claim in one sentence with the numbers. If the signal is too far, the claim is that the data do not show it.

One detector cannot tell a glitch from a signal by $\Phi$ alone. A real search asks for the same template in a second detector, within the light travel time, as in A5. It also measures the false alarm rate from time slides: the second record is shifted by seconds and the coincidences that remain measure the background (lecture 13).

*Optional.* Load the Livingston data with `load_data('L1')`, inject the same signal with `inject_signal` and a merger a few milliseconds earlier (within 10 ms) and search it with the same template. Say whether the two peaks are a coincidence.

*Hint:* `matched_filter(data, template, psd)` on the data without the signal gives the background.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Software

`pycbc` for the filter and the spectrum; `gwcourse_signal` to load the data. The data are the thirty-two seconds of H1 and L1 provided in `labs/data/`.

The signals and the glitches are **simulated**. The binaries of A2 and Part B and the glitches of A4 are models. They are used because the cached data contains no glitch and one real event only.

**References**

- Usman et al. (2016). *The PyCBC search for gravitational waves from compact binary coalescence*.
- LIGO Scientific Collaboration and Virgo Collaboration (2016). *Observation of Gravitational Waves from a Binary Black Hole Merger*.